<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c09-header.png" alt="Nextia Learning · RAG: Building AI That Uses Your Data" width="100%">

# Solution: Search technical documentation

**[Search technical documentation](https://learning.nextia-ai.com/courses/rag/m07/search-technical-documentation/)** · RAG: Building AI That Uses Your Data · Case study · [learning.nextia-ai.com](https://learning.nextia-ai.com)

You compare keyword search, vector search, hybrid search and reranking on real technical documentation: the SQLite 3.53.4 documentation (public domain), 195 pages cut into 4,300 chunks, with 53 labelled questions. You measure recall@k, MRR and hit@1 by kind of question, plus time and cost, and you decide what wins **on this collection**.

| | |
|---|---|
| **Time** | About 2 hours with the lesson page; the notebook ran in about 2 minutes on the test laptop, with the models already downloaded |
| **Needs** | No account and no key. Internet for the setup (about 20 MB of data and two models of about 470 MB each) |
| **Tested** | 2026-10-09 on macOS (Apple silicon), Python 3.12.13, sentence-transformers 6.1.0, CPU only. Colab and Kaggle are not tested yet |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/rag/m07/search-technical-documentation/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/usmanahmed99/nextia-learning-labs/blob/main/C09/M07-L02-search-technical-documentation/search-technical-documentation-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs sentence-transformers 6.1.0, the version of the course, and pandas and matplotlib if they are missing. Colab and Kaggle already have PyTorch, so the cell does not install it again (that would download about 800 MB). It takes up to a minute.

You should see `sentence-transformers 6.1.0`. If you see another version, restart the session (Colab: **Runtime › Restart session**), then run the cell again.

In [ ]:
%pip install -q --disable-pip-version-check sentence-transformers==6.1.0 pandas matplotlib
import sentence_transformers, torch, sqlite3
print("sentence-transformers", sentence_transformers.__version__, "| torch", torch.__version__, "| SQLite", sqlite3.sqlite_version)

## Setup: get the documentation snapshot

The next cell downloads the case study's data from the course's labs repository (about 20 MB) into a folder `sqlite-docs`, and checks the SHA-256 checksum of every file:

- `sqlite-doc-3530400-pages.zip`: 195 pages of the official SQLite 3.53.4 documentation, unchanged. SQLite's documentation is in the public domain.
- `questions.jsonl`: the 53 labelled questions.
- `vectors-e5.npz`: the e5 vectors of the 4,300 chunks, computed with the course's local model at its pinned revision. They save about 2 to 5 minutes of computing.
- `vectors-embed-small.npz`: the vectors of the hosted model `embed-small`, recorded once, so that you need no key.
- `answers_chat-small.jsonl`: answers recorded from `chat-small` (gpt-6-luna on Azure, 2026-10-09).
- `chunks.jsonl.gz`: the chunks, only to compare with the chunks that you build.

You should see `Ready: 6 files checked.` If a file is next to the notebook, the cell uses it.

In [ ]:
# Setup: download the data into sqlite-docs/ and check each file.
import hashlib, os, shutil, subprocess, urllib.request
from pathlib import Path

LABS = "https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/C09/M07-L02-search-technical-documentation/data/"
FILES = {
    "answers_chat-small.jsonl": "97c3596240ab732864f2216e158617499e420388f4da31780b0fa9778d0ec42e",
    "chunks.jsonl.gz": "8e9e40f69fd0420644d86645fd3e43db711693aea46bd312580af5a1dfea0278",
    "questions.jsonl": "6a03203667dfa5cca2e07ace4e363d61311b0b2a386db807e1c4a5b75e2f144e",
    "sqlite-doc-3530400-pages.zip": "22ddd7e72f28b2f629f941f6e3f80763be1b9d9002f58318a9aac7b2e5793aa8",
    "vectors-e5.npz": "ff160073c8325a53063d9e1a64b379330e7fe9c3ad0913afbdaf42de954ca122",
    "vectors-embed-small.npz": "35c5d22251cf5bc4e2d89c9ecde200f7cba2d810c63860a33244f55519cc98d8"
}
HERE = Path.cwd()
if Path(".nextia-notebook").exists():
    os.chdir(".."); HERE = Path.cwd()
WORK = Path("sqlite-docs").resolve()
WORK.mkdir(exist_ok=True); (WORK / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def fetch(name, expected):
    path = WORK / name
    if path.exists() and sha256(path) == expected:
        return
    for local in (HERE / name, HERE / "data" / name):
        if local.exists():
            shutil.copy(local, path); break
    else:
        try:
            urllib.request.urlretrieve(LABS + name, path)
        except OSError:
            if subprocess.run(["curl", "-fsSL", "-o", str(path), LABS + name]).returncode:
                raise SystemExit(f"Could not download {name}. Check your internet connection.")
    if sha256(path) != expected:
        raise SystemExit(f"{name} has the wrong checksum. Delete the folder sqlite-docs and run again.")

for name, expected in FILES.items():
    fetch(name, expected)
os.chdir(WORK)
print(f"Ready: {len(FILES)} files checked.")

## Setup: helpers (1 of 2): parse and chunk

The next cell defines the parser and the chunker of this case study. They follow the course project (the lessons *Parse and normalize*, *Chunk deliberately* and *Preserve metadata*), adapted to HTML pages:

- `parse_page()` skips what SQLite itself marks as navigation (`<div class=nosearch>`), scripts and the syntax diagrams. It keeps each heading with its anchor (for example `#pragma_busy_timeout`), so every chunk links back to its place on the page.
- `chunk_page()` makes one chunk per section, at most 200 words, split between blocks. A long table is split between rows, and each part repeats the header row. The chunk ID is a hash of page, version and position.

Run it. It prints nothing.

In [ ]:
import hashlib
import html
import re
import sqlite3
import statistics
from dataclasses import dataclass, field
from html.parser import HTMLParser

VERSION = "3.53.4"
MAX_WORDS = 200
BLOCK_TAGS = {"p", "pre", "li", "dt", "dd", "blockquote", "div", "table", "ul", "ol", "dl", "hr", "br", "center"}
HEADING_TAGS = {"h1", "h2", "h3", "h4", "h"}   # SQLite's pragma page uses a hidden <h _id=...> per pragma
SKIP_TAGS = {"script", "style", "svg", "head", "title"}


# ---------------------------------------------------------------- parse

@dataclass
class Section:
    heading: str
    anchor: str
    blocks: list = field(default_factory=list)   # (kind, text): kind "text" or "table"


@dataclass
class Page:
    doc_id: str
    title: str
    sections: list


class _PageParser(HTMLParser):
    def __init__(self):
        super().__init__(convert_charrefs=True)
        self.title = ""
        self.in_title = False
        self.skip = 0              # depth inside script/style/svg/head
        self.div_stack = []        # True for a <div class=nosearch>
        self.nosearch = 0
        self.sections = [Section("", "")]
        self.buf = []              # text of the current block
        self.heading = None        # text of the heading being read, or None
        self.anchor = ""           # last <a name=...> or id seen
        self.table = None          # rows of the current table, or None
        self.row = None

    # -- helpers
    def _flush(self):
        text = re.sub(r"\s+", " ", "".join(self.buf)).strip()
        self.buf = []
        if not text:
            return
        if self.row is not None:
            self.row.append(text)
        else:
            self.sections[-1].blocks.append(("text", text))
        self.anchor = ""           # an anchor names the heading right after it, not a later one

    def _hidden(self):
        return self.skip > 0 or self.nosearch > 0

    def handle_starttag(self, tag, attrs):
        a = dict(attrs)
        if tag == "title":
            self.in_title = True
        if tag in SKIP_TAGS and tag != "title":
            self.skip += 1
            return
        if tag == "div":
            ns = "nosearch" in (a.get("class") or "").split()
            self.div_stack.append(ns)
            self.nosearch += ns
        if self._hidden():
            return
        if tag == "a" and a.get("name"):
            self.anchor = a["name"]
        if a.get("id") and tag in HEADING_TAGS:
            self.anchor = a["id"]
        if tag == "h" and a.get("_id"):
            self.anchor = a["_id"]
        if tag in HEADING_TAGS:
            anchor = self.anchor          # the text before the heading may not be flushed yet
            self._flush()
            self.anchor = anchor
            self.heading = []
            return
        if tag == "table":
            self._flush()
            if self.table is None:
                self.table = []
            return
        if tag == "tr" and self.table is not None:
            self._flush()
            self.row = []
            return
        if tag in ("td", "th") and self.row is not None:
            self._flush()
            return
        if tag in BLOCK_TAGS:
            self._flush()

    def handle_endtag(self, tag):
        if tag == "title":
            self.in_title = False
        if tag in SKIP_TAGS and tag != "title":
            self.skip = max(0, self.skip - 1)
            return
        if tag == "div":
            if self.div_stack:
                self.nosearch -= self.div_stack.pop()
            if not self._hidden():
                self._flush()
            return
        if self._hidden():
            return
        if tag in HEADING_TAGS and self.heading is not None:
            text = re.sub(r"\s+", " ", "".join(self.heading)).strip()
            text = re.sub(r"^(\d+\.)+\s*", "", text)          # "4.2. Foo" -> "Foo"
            self.heading = None
            if text:
                anchor = self.anchor or re.sub(r"[^a-z0-9]+", "_", text.lower()).strip("_")
                used = {x.anchor for x in self.sections}
                base, n = anchor, 2
                while anchor in used:
                    anchor, n = f"{base}_{n}", n + 1
                self.sections.append(Section(text, anchor))
                self.anchor = ""
            return
        if tag in ("td", "th") and self.row is not None:
            self._flush()
            return
        if tag == "tr" and self.row is not None:
            self._flush()
            if self.row:
                self.table.append(" | ".join(self.row))
            self.row = None
            return
        if tag == "table" and self.table is not None:
            self._flush()
            if self.row:
                self.table.append(" | ".join(self.row))
            if self.table:
                self.sections[-1].blocks.append(("table", "\n".join(self.table)))
            self.table, self.row = None, None
            return
        if tag in BLOCK_TAGS:
            self._flush()

    def handle_data(self, data):
        if self.in_title:
            self.title += data
            return
        if self._hidden():
            return
        if self.heading is not None:
            self.heading.append(data)
        else:
            self.buf.append(data)


FAQ_Q = re.compile(r"^\((\d+)\)\s+(.*)")


def parse_page(doc_id: str, raw_html: str) -> Page:
    """One SQLite documentation page -> its title and sections, in reading order."""
    p = _PageParser()
    p.feed(raw_html)
    p.close()
    p._flush()
    title = re.sub(r"\s+", " ", html.unescape(p.title)).strip()
    sections = []
    for s in p.sections:
        # The FAQ page has no headings: each question is a bold "(N) ..." paragraph. Make it a heading.
        if doc_id == "faq":
            for kind, text in s.blocks:
                m = FAQ_Q.match(text)
                if m and kind == "text":
                    sections.append(Section(m.group(2), f"q{m.group(1)}"))
                else:
                    (sections[-1] if sections else s).blocks.append((kind, text))
            if not sections:
                sections.append(s)
            continue
        if s.blocks or s.heading:
            sections.append(s)
    sections = [s for s in sections if s.blocks]
    return Page(doc_id, title, sections)


# ---------------------------------------------------------------- chunk

def chunk_id(doc_id: str, version: str, start: int, end: int) -> str:
    return hashlib.sha256(f"{doc_id}|{version}|{start}|{end}".encode("utf-8")).hexdigest()[:12]


def _pieces(kind: str, text: str, pos: int, max_words: int) -> list[tuple]:
    """A block, or its pieces when it is longer than max_words: a table between rows (the later pieces
    keep the header row), other text between words. Each piece is (kind, text, start, end, header)."""
    if len(text.split()) <= max_words:
        return [(kind, text, pos, pos + len(text), "")]
    sep = "\n" if kind == "table" and "\n" in text else " "
    units = text.split(sep)
    header = units[0] if sep == "\n" else ""
    pieces, current, count, offset, start = [], [], 0, pos, pos
    for u in units:
        n = len(u.split())
        if current and count + n > max_words:
            body = sep.join(current)
            pieces.append((kind, body, start, start + len(body), header if pieces else ""))
            start, current, count = start + len(body) + 1, [], 0
        current.append(u)
        count += n
    body = sep.join(current)
    pieces.append((kind, body, start, start + len(body), header if pieces else ""))
    return pieces


def chunk_page(page: Page, version: str = VERSION, max_words: int = MAX_WORDS) -> list[dict]:
    """Structure chunks: one per section, split between blocks at max_words, tables never split.

    Positions are character offsets in the page's parsed text (all sections, blocks joined by
    newlines), so a chunk can be traced back to its place on the page.
    """
    chunks, offset = [], 0
    for s_index, s in enumerate(page.sections):
        part, part_start = [], None
        pos = offset + len(s.heading) + 1
        spans = []
        for kind, text in s.blocks:
            spans += _pieces(kind, text, pos, max_words)
            pos += len(text) + 1

        def emit(blocks, continued):
            start, end = blocks[0][2], blocks[-1][3]
            body = "\n".join(b[1] for b in blocks)
            head = s.heading if s.heading else ""
            prefix = page.title + (f"\n{head}" if head else "")
            if blocks[0][4]:                       # a later part of a long table: repeat its header row
                prefix += f"\n{blocks[0][4]}"
            chunks.append({
                "chunk_id": chunk_id(page.doc_id, version, start, end),
                "doc_id": page.doc_id, "version": version, "title": page.title,
                "section": head, "anchor": s.anchor, "section_index": s_index,
                "position": len(chunks), "start": start, "end": end, "continued": continued,
                "text": f"{prefix}\n{body}",
                "url": f"https://sqlite.org/{page.doc_id}.html" + (f"#{s.anchor}" if s.anchor else ""),
            })

        count, continued = 0, False
        for b in spans:
            words = len(b[1].split())
            if part and count + words > max_words:
                emit(part, continued)
                part, count, continued = [], 0, True
            part.append(b)
            count += words
        if part:
            emit(part, continued)
        offset = pos
    return chunks

## Setup: helpers (2 of 2): search, measure, check

The next cell defines the search methods of the course and the measures:

- `KeywordIndex`: SQLite FTS5 with BM25. `dense_search()`: cosine similarity with NumPy. `rrf()`: reciprocal rank fusion with the constant 60.
- `scores_with_evidence()`: hit@k, recall@k and the reciprocal rank of the first relevant chunk (in the top 10) for one question.
- `check_claims()`: the citation checks that code can do.
- `expect()` and `expect_hash()` check your answers to the tasks without showing them.

Run it. It prints nothing.

In [ ]:
# ---------------------------------------------------------------- search

def words(text: str) -> list[str]:
    return list(dict.fromkeys(w.lower() for w in re.findall(r"\w+", text)))


def fts_query(text: str) -> str:
    """Every word quoted (so FTS5 never reads a word as an operator), joined by OR: the course's query."""
    return " OR ".join(f'"{w}"' for w in words(text))


class KeywordIndex:
    """SQLite FTS5 with BM25 over the chunks (in memory)."""

    def __init__(self, chunks: list[dict]):
        self.conn = sqlite3.connect(":memory:")
        self.conn.execute("CREATE VIRTUAL TABLE kw USING fts5(item_id UNINDEXED, text, "
                          "tokenize = 'unicode61 remove_diacritics 2')")
        self.conn.executemany("INSERT INTO kw (item_id, text) VALUES (?, ?)",
                              [(c["chunk_id"], c["text"]) for c in chunks])

    def search(self, query: str, k: int = 20) -> list[tuple[str, float]]:
        q = fts_query(query)
        if not q:
            return []
        rows = self.conn.execute("SELECT item_id, bm25(kw) FROM kw WHERE kw MATCH ? "
                                 "ORDER BY bm25(kw), item_id LIMIT ?", (q, k)).fetchall()
        return [(i, round(-s, 4)) for i, s in rows]


def dense_search(query_vector, matrix, ids: list[str], k: int = 20) -> list[tuple[str, float]]:
    """Brute force: the dot product of normalised vectors is their cosine similarity."""
    import numpy as np
    scores = matrix @ query_vector
    order = np.lexsort((np.array(ids), -scores))[:k]
    return [(ids[i], round(float(scores[i]), 4)) for i in order]


def rrf(rankings: list[list[tuple[str, float]]], k: int = 20, constant: int = 60) -> list[tuple[str, float]]:
    """Reciprocal rank fusion: each list gives 1 / (60 + rank); the sums are sorted."""
    scores: dict[str, float] = {}
    for ranking in rankings:
        for rank, (item, _) in enumerate(ranking, 1):
            scores[item] = scores.get(item, 0.0) + 1.0 / (constant + rank)
    order = sorted(scores.items(), key=lambda x: (-x[1], x[0]))
    return [(i, round(s, 6)) for i, s in order[:k]]


# ---------------------------------------------------------------- measure

def covers(chunk: dict, passage: dict) -> bool:
    """A chunk is relevant to a labelled passage when it comes from the same page and section and, for a
    long section, contains the passage's evidence text."""
    return (chunk["doc_id"] == passage["doc_id"] and chunk["anchor"] == passage["anchor"]
            and (not passage.get("evidence") or re.search(passage["evidence"], chunk["text"]) is not None))


def scores_with_evidence(ranked: list[dict], passages: list[dict], k: int) -> dict:
    """For one question: hit@k (a relevant chunk in the top k), recall@k (the share of its labelled
    passages found in the top k), the reciprocal rank of the first relevant chunk within the top 10, and
    that rank (None if no relevant chunk is in the list)."""
    top = ranked[:k]
    found = [p for p in passages if any(covers(c, p) for c in top)]
    first = next((i for i, c in enumerate(ranked) if any(covers(c, p) for p in passages)), None)
    return {"hit": float(bool(found)), "recall": len(found) / len(passages),
            "rr": 0.0 if first is None or first >= 10 else 1.0 / (first + 1),
            "first_rank": None if first is None else first + 1}


def mean(values) -> float:
    values = list(values)
    return round(statistics.fmean(values), 3) if values else float("nan")


# ---------------------------------------------------------------- answers

def format_passages(chunks: list[dict]) -> str:
    """The passages for the prompt: ID, page, version and section first, so that every claim can cite them."""
    return "\n\n".join(f"[{c['chunk_id']}] {c['title']} | SQLite {c['version']} documentation, {c['doc_id']}.html | "
                       f"section: {c['section'] or '-'}\n{c['text']}" for c in chunks)


STOP = set("a an the and or of to in on for by with is are be it this that as at from if not no can its".split())


def check_claims(answer: dict, context_ids: list[str], chunks: dict) -> list[dict]:
    """What code can check, as in the course: each claim cites at least one passage, every cited ID was in
    the context, and at least half of the claim's content words appear in the cited passages. It cannot
    check that the claim is true; a person reads the passage for that."""
    out = []
    for claim in answer.get("claims", []):
        problems = []
        ids = claim.get("chunk_ids", [])
        if not ids:
            problems.append("no citation")
        bad = [i for i in ids if i not in context_ids]
        problems += [f"not_in_context:{i}" for i in bad]
        text = " ".join(chunks[i]["text"] for i in ids if i in chunks).lower()
        content = [w for w in re.findall(r"[a-z0-9_]+", claim["text"].lower()) if w not in STOP]
        share = sum(1 for w in content if w in text) / len(content) if content else 1.0
        if share < 0.5:
            problems.append(f"words {share:.2f}")
        out.append({"claim": claim["text"], "ids": ids, "words_found": round(share, 2), "problems": problems})
    return out


# ---------------------------------------------------------------- task checks
import json as _json

def fingerprint(value):
    def norm(v):
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 4)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(_json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif isinstance(yours, (int, float)) and abs(yours - expected) <= tolerance * max(1, abs(expected)):
        print(f"Check passed: {what} is {round(yours, 6)}.")
    elif isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. The situation

Dana runs the developer support desk of a small company whose product stores its data in SQLite. Her team answers about 40 questions a week, half from people who paste an exact name (`SQLITE_BUSY`, `PRAGMA journal_size_limit`, `.once`), and half from people who describe a problem in their own words ("the file did not get smaller"). She wants a search box over the official documentation that finds the right section for both kinds, and later an assistant that answers with citations.

Her question to you: **which search method should the team use, and what does it cost?** The [lesson page](https://learning.nextia-ai.com/courses/rag/m07/search-technical-documentation/) tells the whole story; this notebook runs every step.

## 2. The data: parse the pages and keep the provenance

The next cell opens the 195 pages, parses each one and cuts it into chunks. Then it compares your chunks with the shipped `chunks.jsonl.gz`. They should be identical, because the parser only uses Python's standard library.

> **Check.** You should see `195 pages, 2231 sections, 4300 chunks` and `Your chunks are identical to the shipped ones.`

In [ ]:
import gzip, zipfile
pages, chunks = [], []
with zipfile.ZipFile("sqlite-doc-3530400-pages.zip") as z:
    for name in sorted(z.namelist()):
        page = parse_page(name.split("/")[-1][:-5], z.read(name).decode("utf-8", errors="replace"))
        pages.append(page)
        chunks += chunk_page(page)
print(f"{len(pages)} pages, {sum(len(p.sections) for p in pages)} sections, {len(chunks)} chunks")
mine = "".join(_json.dumps(c, ensure_ascii=False) + "\n" for c in chunks).encode("utf-8")
if hashlib.sha256(mine).hexdigest() == "e8be8191919e2afccb5d1e7b583062b1365a8a057d18f326efa7274d40591463":
    print("Your chunks are identical to the shipped ones.")
else:
    print("Your chunks differ from the shipped ones (another Python version?). The notebook uses the shipped chunks.")
    chunks = [_json.loads(x) for x in gzip.decompress(Path("chunks.jsonl.gz").read_bytes()).decode().splitlines()]
by_id = {c["chunk_id"]: c for c in chunks}
ids = [c["chunk_id"] for c in chunks]
sizes = sorted(len(c["text"].split()) for c in chunks)
print(f"{sum(sizes):,} words; median chunk {sizes[len(sizes) // 2]} words; longest {sizes[-1]}")

Every chunk keeps its page, its section, its anchor and a link to the exact place in the official documentation. The next cell shows one.

> **Check.** You should see the chunk of `PRAGMA busy_timeout`, with the URL `https://sqlite.org/pragma.html#pragma_busy_timeout`.

In [ ]:
c = next(c for c in chunks if c["anchor"] == "pragma_busy_timeout")
print({k: c[k] for k in ("chunk_id", "doc_id", "version", "section", "url")})
print(c["text"][:400])

### Two traps in the data

Real documentation is not clean. The next cell finds two traps that matter for search and for answers:

1. **Near-duplicate pages.** `fileformat.html` and `fileformat2.html` describe the same file format, almost word for word. Both are in the snapshot, as on the website.
2. **An old version inside the current docs.** `c_interface.html` is titled *The C language interface to SQLite Version 2*. It is in the 3.53.4 documentation, but it describes an interface from more than 20 years ago.

> **Check.** You should see that more than 90% of the `fileformat` chunks have an identical twin in `fileformat2`, and the title with "Version 2".

In [ ]:
def body(c):
    return c["text"].split("\n", 1)[-1]
f1 = {body(c) for c in chunks if c["doc_id"] == "fileformat"}
f2 = {body(c) for c in chunks if c["doc_id"] == "fileformat2"}
print(f"fileformat: {len(f1)} chunks; identical text in fileformat2: {len(f1 & f2)} ({len(f1 & f2) / len(f1):.0%})")
print(next(p.title for p in pages if p.doc_id == "c_interface"))

## 3. The questions and how they were labelled

The 53 questions have four kinds: **identifier** (an exact name in the question, 18, of which 5 are only the name), **paraphrase** (the same idea in other words, 15), **how-to** (a task in the user's words, 15) and **unanswerable** (5: another product, a page not in the snapshot, or a made-up name).

The labels were made in two steps, described on the lesson page: the author wrote each question from one section and labelled the sections he already knew (82 passages); then the top 5 of every method were pooled and read (637 chunks), which added 40 passages. A relevant passage is a section of a page; in a long section, only the chunks that contain an `evidence` text count.

> **Check.** You should see the counts by kind, and the question `D19` with 4 relevant passages.

In [ ]:
questions = [_json.loads(x) for x in Path("questions.jsonl").read_text().splitlines()]
scored = [q for q in questions if q["relevant"]]
from collections import Counter
print(Counter(q["kind"] for q in questions))
print("labelled passages:", Counter(r["source"] for q in questions for r in q["relevant"]))
q = next(q for q in questions if q["id"] == "D19")
print(q["question"])
for r in q["relevant"]:
    print("  ", r["doc_id"], "#", r["anchor"], "| evidence:", r["evidence"], "|", r["source"])

## 4. The baseline: keyword search (BM25)

Start with the simplest method, as in [A keyword-search baseline](https://learning.nextia-ai.com/courses/rag/m01/a-keyword-baseline/): SQLite's own full-text search, FTS5 with BM25, over the 4,300 chunks. The next cell builds the index, runs every question and keeps the top 20.

> **Predict.** Will keyword search find the identifier questions (`What is SQLITE_BUSY_SNAPSHOT?`) better or worse than the how-to questions ("How do I make the text of my documents searchable by words?")? Write your guess.

In [ ]:
import time
import numpy as np

def measure(rankings, qs=scored):
    """hit@1, hit@5, recall@5, recall@10 and MRR@10 over the questions with relevant passages."""
    rows = []
    for q in qs:
        ranked = [by_id[i] for i, _ in rankings[q["id"]]]
        rows.append({k: scores_with_evidence(ranked, q["relevant"], k) for k in (1, 5, 10)})
    return {"hit@1": mean(r[1]["hit"] for r in rows), "hit@5": mean(r[5]["hit"] for r in rows),
            "recall@5": mean(r[5]["recall"] for r in rows), "recall@10": mean(r[10]["recall"] for r in rows),
            "MRR@10": mean(r[10]["rr"] for r in rows)}

def by_kind(rankings):
    return {kind: measure(rankings, [q for q in scored if q["kind"] == kind])["hit@5"]
            for kind in ("identifier", "paraphrase", "howto")}

t = time.perf_counter(); keyword = KeywordIndex(chunks); print(f"index built in {time.perf_counter() - t:.2f} s")
runs, times = {}, {}
t = time.perf_counter()
runs["bm25"] = {q["id"]: keyword.search(q["question"], 20) for q in questions}
times["bm25"] = (time.perf_counter() - t) / len(questions)
print("bm25", measure(runs["bm25"]))
print("hit@5 by kind:", by_kind(runs["bm25"]))

> **Check.** You should see `hit@5` 0.708 and, by kind, identifier 0.944, paraphrase 0.533, how-to 0.6. Keywords are strong on exact names and weak when the user's words differ from the documentation's words.

## 5. Vector search with the local model (e5)

The next cell loads the precomputed e5 vectors. It checks that they were made with the same model and revision as the questions, because vectors of two models cannot be compared ([Embeddings for search](https://learning.nextia-ai.com/courses/rag/m03/embeddings-for-search/)). Then it embeds the 53 questions on your computer and searches.

To compute the chunk vectors yourself, set `RECOMPUTE = True`. It took 108 seconds on the test laptop (2 threads); expect longer on free Colab.

> **Check.** You should see `e5 vectors: 4300 x 384` and `hit@5` 0.729.

In [ ]:
from sentence_transformers import SentenceTransformer, CrossEncoder
torch.set_num_threads(2)
E5 = ("intfloat/multilingual-e5-small", "614241f622f53c4eeff9890bdc4f31cfecc418b3")
RECOMPUTE = False
saved = np.load("vectors-e5.npz")
assert (str(saved["model"]), str(saved["revision"])) == E5, "These vectors were made with another model."
assert [str(x) for x in saved["chunk_ids"]] == ids, "The vectors do not belong to these chunks."
e5 = SentenceTransformer(E5[0], revision=E5[1], device="cpu")
if RECOMPUTE:
    t = time.perf_counter()
    e5_matrix = e5.encode(["passage: " + c["text"] for c in chunks], batch_size=16, normalize_embeddings=True)
    print(f"embedded {len(chunks)} chunks in {time.perf_counter() - t:.0f} s")
else:
    e5_matrix = saved["vectors"].astype(np.float32)
print("e5 vectors:", " x ".join(map(str, e5_matrix.shape)))
t = time.perf_counter()
e5_q = {q["id"]: e5.encode(["query: " + q["question"]], normalize_embeddings=True)[0] for q in questions}
runs["dense-e5"] = {q["id"]: dense_search(e5_q[q["id"]], e5_matrix, ids, 20) for q in questions}
times["dense-e5"] = (time.perf_counter() - t) / len(questions)
print("dense-e5", measure(runs["dense-e5"]))
print("hit@5 by kind:", by_kind(runs["dense-e5"]))

## 6. Vector search with the hosted model (embed-small, recorded)

`embed-small` is OpenAI's text-embedding-3-small, deployed on Azure for the course. Its vectors of the chunks and of the 53 questions were recorded once (2026-10-09: 828,297 tokens, US$0.0166). The next cell replays them. A new question would need a key, so this method is replay-only here.

> **Check.** You should see `hit@5` 0.917.

In [ ]:
rec = np.load("vectors-embed-small.npz")
assert [str(x) for x in rec["chunk_ids"]] == ids
es_matrix = rec["vectors"].astype(np.float32)
es_q = dict(zip((str(k) for k in rec["question_keys"]), rec["question_vectors"].astype(np.float32)))
def es_key(text):
    return hashlib.sha256(f"embed-small\x00{text}".encode("utf-8")).hexdigest()[:16]
runs["dense-embed-small"] = {q["id"]: dense_search(es_q[es_key(q["question"])], es_matrix, ids, 20) for q in questions}
print(str(rec["revision"]))
print("dense-embed-small", measure(runs["dense-embed-small"]))
print("hit@5 by kind:", by_kind(runs["dense-embed-small"]))

## 7. Hybrid search and reranking

Hybrid search fuses the BM25 top 20 and a vector top 20 with reciprocal rank fusion ([Lexical, dense and hybrid search](https://learning.nextia-ai.com/courses/rag/m03/lexical-dense-and-hybrid-search/)). Reranking rereads the hybrid top 20 with the course's cross-encoder ([Reranking and query changes](https://learning.nextia-ai.com/courses/rag/m03/reranking-and-query-changes/)). Reranking is the slow step: about 0.7 s per question on a laptop, more on free Colab.

> **Predict.** On Larkfield's policies, hybrid search with e5 scored *below* e5 alone. Will it do the same here?

In [ ]:
runs["hybrid-e5"] = {q["id"]: rrf([runs["bm25"][q["id"]], runs["dense-e5"][q["id"]]], 20) for q in questions}
runs["hybrid-embed-small"] = {q["id"]: rrf([runs["bm25"][q["id"]], runs["dense-embed-small"][q["id"]]], 20) for q in questions}
reranker = CrossEncoder("cross-encoder/mmarco-mMiniLMv2-L12-H384-v1", revision="1427fd652930e4ba29e8149678df786c240d8825", device="cpu")

def rerank(question, candidate_ids, k=20):
    cand = [by_id[i] for i in candidate_ids]
    scores = reranker.predict([(question, c["text"]) for c in cand], batch_size=16)
    order = sorted(range(len(cand)), key=lambda i: (-float(scores[i]), cand[i]["chunk_id"]))
    return [(cand[i]["chunk_id"], round(float(scores[i]), 4)) for i in order[:k]]

t = time.perf_counter()
runs["rerank"] = {q["id"]: rerank(q["question"], [i for i, _ in runs["hybrid-e5"][q["id"]]]) for q in questions}
times["rerank"] = (time.perf_counter() - t) / len(questions)
for m in ("hybrid-e5", "hybrid-embed-small", "rerank"):
    print(m, measure(runs[m]))
print(f"rerank: {times['rerank'] * 1000:.0f} ms per question on this computer")

## 8. The fair comparison

Every method ran on the same 4,300 chunks, the same 48 scored questions, the same labels and the same measures. The next cell puts them in one table, then hit@5 by kind, then recall@k as a chart.

> **Check.** In the first table, `dense-embed-small` has the highest hit@5 (0.917) and MRR@10 (0.789); `rerank` is the best local method (0.875).

In [ ]:
import pandas as pd
order = ["bm25", "dense-e5", "dense-embed-small", "hybrid-e5", "hybrid-embed-small", "rerank"]
table = pd.DataFrame({m: measure(runs[m]) for m in order}).T
print(table.round(3).to_string())
print()
print(pd.DataFrame({m: by_kind(runs[m]) for m in order}).T.round(3).to_string())

In [ ]:
import matplotlib.pyplot as plt
ks = [1, 3, 5, 10, 20]
fig, ax = plt.subplots(figsize=(7, 4))
for m in order:
    r = [mean(scores_with_evidence([by_id[i] for i, _ in runs[m][q["id"]]], q["relevant"], k)["recall"] for q in scored) for k in ks]
    ax.plot(ks, r, marker="o", label=m)
ax.set_xscale("log"); ax.set_xticks(ks); ax.set_xticklabels(ks)
ax.set_xlabel("k (chunks returned)"); ax.set_ylabel("recall@k"); ax.set_ylim(0, 1); ax.legend(fontsize=8); ax.grid(alpha=.3)
plt.show()

### Is the difference real?

With 48 questions, one question is 0.021 of hit@5. The next cell counts, for two methods, the questions that only one of them finds in its top 5, and draws 10,000 bootstrap samples of the questions for a 95% interval of the difference.

> **Check.** For `dense-embed-small` against `rerank`, you should see 5 questions against 3, and an interval that contains 0: the difference is not proof. For `rerank` against `bm25`, 9 against 1, and an interval above 0.

In [ ]:
import random
def hits(m):
    return {q["id"]: scores_with_evidence([by_id[i] for i, _ in runs[m][q["id"]]], q["relevant"], 5)["hit"] for q in scored}
def compare(a, b, draws=10000, seed=20261009):
    ha, hb, qids = hits(a), hits(b), [q["id"] for q in scored]
    rng = random.Random(seed)
    diffs = sorted(sum(ha[i] - hb[i] for i in s) / len(s) for s in ([rng.choice(qids) for _ in qids] for _ in range(draws)))
    only_a = [i for i in qids if ha[i] and not hb[i]]; only_b = [i for i in qids if hb[i] and not ha[i]]
    print(f"{a} vs {b}: only {a} {only_a}; only {b} {only_b}; 95% interval of the hit@5 difference "
          f"[{diffs[int(draws * .025)]:+.3f}, {diffs[int(draws * .975) - 1]:+.3f}]")
compare("dense-embed-small", "rerank")
compare("rerank", "hybrid-e5")
compare("rerank", "bm25")

## 9. Time and cost

Quality is one column of the decision. The next cell adds the time per question measured on your computer. The indexing cost of `embed-small` comes from the recording: 828,297 tokens at US$0.02 per million, US$0.0166, about 10 minutes with the shared deployment's rate limits. The local models cost nothing per call but need about 1 GB of downloads and a CPU.

In [ ]:
print(f"bm25 query:            {times['bm25'] * 1000:7.1f} ms")
print(f"e5 query + search:     {times['dense-e5'] * 1000:7.1f} ms")
print(f"rerank (20 chunks):    {times['rerank'] * 1000:7.1f} ms")
print("embed-small query:     about 180-770 ms per call (one network call; recorded 2026-10-09)")

## 10. Failure case: the how-to question that no method ranks first

> **Warning.** "How do I check a database file for corruption?" sounds easy. The answer is `PRAGMA integrity_check`. Look at what the rerank top 5 contains.

In [ ]:
q = next(q for q in questions if q["id"] == "D41")
for rank, (cid, score) in enumerate(runs["rerank"]["D41"][:5], 1):
    c = by_id[cid]
    print(rank, f"{score:7.3f}", c["doc_id"], "|", c["section"][:60])
print("first relevant rank:", {m: scores_with_evidence([by_id[i] for i, _ in runs[m]["D41"]], q["relevant"], 20)["first_rank"] for m in order})

The top 5 is full of *How To Corrupt An SQLite Database File*: the page with the most words about corruption. The pragma's own section says "low-level formatting and consistency check", not "corruption". Words that match the question are not always words that answer it. Look at the ranks: both hybrid lists have the FAQ answer ("You can use PRAGMA integrity_check ...") first, and the reranker moves it down to 8th. A second model is not always a better judge. The recorded answer with the rerank top 5 abstained: "The documentation passages do not say how to check a database file for corruption."

## 11. End to end: a new question with a cited answer

A new question that is not in the question set. The next cell runs the rerank pipeline on it with your local models, then replays the answer that `chat-small` gave on 2026-10-09 to the same top 5, and checks each claim in code.

> **Predict.** The right answer is a busy timeout (`PRAGMA busy_timeout`). Will the top 5 contain it?

In [ ]:
recorded = {e["qid"]: e for e in map(_json.loads, Path("answers_chat-small.jsonl").read_text().splitlines())}
new = recorded["NEW1"]
question = new["question"]
qv = e5.encode(["query: " + question], normalize_embeddings=True)[0]
top = [i for i, _ in rerank(question, [i for i, _ in rrf([keyword.search(question, 20), dense_search(qv, e5_matrix, ids, 20)], 20)], 5)]
print(question, "\n")
for i in top:
    print(" ", i, by_id[i]["url"], "|", by_id[i]["title"][:50])
if top != new["passages"]:
    print("\nNote: recorded with passages", new["passages"], "; yours:", top, ". The recorded answer is replayed.")
answer = _json.loads(new["response"])
print("\nchat-small (gpt-6-luna on Azure, recorded", new["at"][:10] + "):", answer["answer"])
for r in check_claims(answer, top, by_id):
    print(" -", r["claim"], r["ids"], "problems:", r["problems"] or "none")

Every claim passes the code checks, and the answer is still poor. One passage is from the **Version 2** C interface page and one from the **Tcl** interface. The answer that a developer needs, `PRAGMA busy_timeout` or `sqlite3_busy_timeout()`, is not in the top 5. A citation shows where a claim comes from; it does not show that it is the right source for this version and this interface. That is why the case study cites the version (3.53.4) and the page with each answer.

## 12. Your turn

### Task 1: where do keywords win against the best model?

Make the sorted list `bm25_only` of the IDs of the questions where BM25 has a relevant chunk in its top 5 and `dense-embed-small` does not. Use `hits()` from section 8.

In [ ]:
h_bm25, h_es = hits("bm25"), hits("dense-embed-small")
bm25_only = sorted(i for i in h_bm25 if h_bm25[i] and not h_es[i])
print(bm25_only)

In [ ]:
expect_hash("bm25_only", bm25_only, '07e322d3130ef8dd')

### Task 2: reciprocal rank fusion by hand

For the question about `.once`, the first relevant chunk is 16th for BM25 and 3rd for `embed-small`. Compute its RRF score with the constant 60, as `rrf()` does.

In [ ]:
score = 1 / (60 + 16) + 1 / (60 + 3)
print(round(score, 6))

In [ ]:
expect("the RRF score", score, 0.029031, tolerance=1e-4)

### Task 3: change one thing: rerank 40 candidates instead of 20

Reranking can only reorder what the first stage found. Give it more: rerank the hybrid **top 40** (BM25 top 40 and e5 top 40, fused) for the 30 paraphrase and how-to questions, and compare hit@5 and the time with the 20-candidate run. Set `pairs_scored` to the number of (question, chunk) pairs that the cross-encoder read.

> **Predict.** Will hit@5 go up, down or stay the same? By how much will the time grow?

In [ ]:
subset = [q for q in scored if q["kind"] in ("paraphrase", "howto")]
t = time.perf_counter()
wide = {}
pairs_scored = 0
for q in subset:
    cand = [i for i, _ in rrf([keyword.search(q["question"], 40), dense_search(e5_q[q["id"]], e5_matrix, ids, 40)], 40)]
    pairs_scored += len(cand)
    wide[q["id"]] = rerank(q["question"], cand)
seconds = time.perf_counter() - t
print("40 candidates:", measure(wide, subset)["hit@5"], f"({seconds / len(subset) * 1000:.0f} ms per question)")
print("20 candidates:", measure(runs["rerank"], subset)["hit@5"], f"({times['rerank'] * 1000:.0f} ms per question)")

In [ ]:
expect("pairs_scored", pairs_scored, 1200)

## Recap

- On this collection, the hosted `embed-small` alone finds the most (hit@5 0.917, MRR@10 0.789). The best local method is hybrid e5 + rerank (0.875). Keyword search alone is the weakest (0.708), but it is the strongest on exact names.
- Unlike Larkfield's policies, hybrid search with e5 **helps** here (0.729 → 0.792), and hybrid with `embed-small` **hurts** (0.917 → 0.833). The same method helps with one model and hurts with another: measure on your own questions.
- With 48 questions, most differences between the strong methods are not proof. Count the questions that only one method finds.
- A cited answer can pass every code check and still come from the wrong version or interface. Cite the version, and keep a person in the loop.

Write your decision for Dana in the lesson page's portfolio section.

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/rag/m07/search-technical-documentation/)

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The documentation is SQLite's own, in the public domain; the questions and labels were written for this course.